In [1]:
import pandas as pd
import numpy as np

In [12]:
tracks = pd.read_csv("../data/track_features.csv")
print(tracks.shape)
print(tracks.columns.tolist())
tracks.head()

(114000, 21)
['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [ ]:
plays = pd.read_csv("../data/playlist_interactions.csv", nrows=5, skipinitialspace=True)  #had spaces as the first letter because of which columns were being showed with quotes initially therefore skipinitialspace
print(plays.columns.tolist())
plays

['user_id', 'artistname', 'trackname', 'playlistname']


,user_id,artistname,trackname,playlistname
0,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,(The Angels Wanna Wear My) Red Shoes,HARD ROCK 2010
1,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,"(What's So Funny 'Bout) Peace, Love And Unders...",HARD ROCK 2010
2,9cc0cfd4d7d7885102480dd99e7a90d6,Tiffany Page,7 Years Too Late,HARD ROCK 2010
3,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,Accidents Will Happen,HARD ROCK 2010
4,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,Alison,HARD ROCK 2010


In [8]:
cols = ["user_id", "artistname", "trackname"]
plays = pd.read_csv("../data/playlist_interactions.csv",
                    usecols=cols,
                    skipinitialspace=True,
                    on_bad_lines="skip")
print(plays.shape)
plays.head(3)

(12901979, 3)


,user_id,artistname,trackname
0,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,(The Angels Wanna Wear My) Red Shoes
1,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,"(What's So Funny 'Bout) Peace, Love And Unders..."
2,9cc0cfd4d7d7885102480dd99e7a90d6,Tiffany Page,7 Years Too Late


In [ ]:
#on_bad_lines=skip(last code), this code tells us how many rows were dropped because of that
#0.005 % dropped(error remcved and negligible rows only removed)
raw_lines = sum(1 for _ in open("../data/playlist_interactions.csv", encoding="utf-8", errors="ignore")) - 1
print(raw_lines, "raw lines |", len(plays), "loaded |", raw_lines - len(plays), "dropped")

12902577 raw lines | 12901979 loaded | 598 dropped


In [ ]:
#seeing the corrupt rows- quotes inside the names which make the data value end early than it is supposed to be and gives multiple columns
import csv

bad = []
with open("../data/playlist_interactions.csv", encoding="utf-8", errors="ignore") as f:
    for i, row in enumerate(csv.reader(f)):
        if len(row) != 4:
            bad.append((i, len(row), row[:6]))
        if len(bad) == 5:
            break

for b in bad:
    print(b)

(8622, 3, ['c50566d83fba17b20697039d5824db78', 'SNAP!', 'Rhythm Is A Dancer - Original 12",Everything at once"'])
(14734, 5, ['7511e45f2cc6f6e609ae46c15506538c', 'Glenn Gould', 'Kyllikki" - Three Lyric Pieces for Piano', ' Op. 41 - II. Andantino"', 'Instrumenal - Home Listens'])
(15930, 6, ['7511e45f2cc6f6e609ae46c15506538c', 'Clint Mansell', "We're Not Programs", ' GERTY', ' We\'re People""', 'Studying'])
(18824, 5, ['50346e4190d1707ebc6b39a95f86927a', 'Charles CJ" Hilton', ' Jr./Raphael Saadiq/Stevie Wonder"', 'Never Give You Up', 'Samedi matin'])
(19142, 3, ['1ed9910b0db7fcb779ec65b2ded4892f', 'Sandy B', 'Make the World Go Round - Deep Dish Vocal 12",Soulful vocal house"'])


In [16]:
pairs = plays[["artistname", "trackname"]].drop_duplicates()
print("unique artist/track pairs:", len(pairs))
print("unique users:", plays["user_id"].nunique())

unique artist/track pairs: 2823960
unique users: 15918


In [22]:
import re

def normalise(text):
    if not isinstance(text, str):
        return ""
    t = text.lower()
    t = re.sub(r"\(.*?\)|\[.*?\]", " ", t)              # drop (feat. X), [Remastered]
    t = re.sub(r"\s-\s.*$", " ", t)                     # drop " - Remastered 2011"
    t = re.sub(r"\b(feat|ft|featuring|with)\b.*$", " ", t)
    t = t.replace("'", "").replace("\u2019", "")        # don't -> dont
    t = re.sub(r"[^\w\s]", " ", t, flags=re.UNICODE)    # strip remaining punctuation
    t = re.sub(r"\s+", " ", t).strip()                  # collapse spaces
    return t

In [23]:
tests = [
    'Rhythm Is A Dancer - Original 12"',
    "Levitating (feat. DaBaby)",
    "Hello - Remastered 2015",
    "Don't Stop Me Now",
    "Bohemian Rhapsody",
]
for t in tests:
    print(repr(t), "->", repr(normalise(t)))

'Rhythm Is A Dancer - Original 12"' -> 'rhythm is a dancer'
'Levitating (feat. DaBaby)' -> 'levitating'
'Hello - Remastered 2015' -> 'hello'
"Don't Stop Me Now" -> 'dont stop me now'
'Bohemian Rhapsody' -> 'bohemian rhapsody'


In [26]:
tracks["artist_key"] = tracks["artists"].astype(str).str.split(";").str[0].map(normalise)
tracks["track_key"] = tracks["track_name"].map(normalise)
tracks["key"] = tracks["artist_key"] + "|" + tracks["track_key"]

print(tracks["key"].nunique(), "unique keys out of", len(tracks), "rows")
tracks[["artists", "track_name", "key"]].head(3)

77414 unique keys out of 114000 rows


,artists,track_name,key
0,Gen Hoshino,Comedy,gen hoshino|comedy
1,Ben Woodward,Ghost - Acoustic,ben woodward|ghost
2,Ingrid Michaelson;ZAYN,To Begin Again,ingrid michaelson|to begin again


In [25]:
dupes = tracks["key"].value_counts().head(5)
print(dupes)
print()
tracks[tracks["key"] == dupes.index[0]][["track_id", "artists", "track_name", "track_genre"]]

key
chuck berry|run rudolph run                 151
stevie wonder|what christmas means to me     87
the beach boys|little saint nick             78
burna boy|last last                          75
ella fitzgerald|frosty the snowman           69
Name: count, dtype: int64



,track_id,artists,track_name,track_genre
8163,1S4rxDloMtAduogKeiJZmR,Chuck Berry,Run Rudolph Run,blues
8164,03MW3H9B2P7tgpvzG3klNI,Chuck Berry,Run Rudolph Run,blues
8165,52MCmoSCtPRbVN5Njdo6G5,Chuck Berry,Run Rudolph Run,blues
8166,7m4luTtlene5vS6xUvWxRt,Chuck Berry,Run Rudolph Run,blues
8168,3RXAcz7Sa6JDZSxcH1EEQ6,Chuck Berry,Run Rudolph Run,blues
...,...,...,...,...
92043,3iyTXUFlm1YrquUUgXlPM1,Chuck Berry,Run Rudolph Run,rockabilly
92044,0XgmLT6nDInA6w2yxpVsZh,Chuck Berry,Run Rudolph Run,rockabilly
92046,4vJrtcgQoULwkPyFkXNYCx,Chuck Berry,Run Rudolph Run,rockabilly
92049,3IUpuyEMIgt4GvEZ2TqERF,Chuck Berry,Run Rudolph Run,rockabilly


In [28]:
tracks[tracks["key"] == "chuck berry|run rudolph run"][["track_id", "track_name", "track_genre", "danceability", "energy"]].head(20)

,track_id,track_name,track_genre,danceability,energy
8163,1S4rxDloMtAduogKeiJZmR,Run Rudolph Run,blues,0.647,0.876
8164,03MW3H9B2P7tgpvzG3klNI,Run Rudolph Run,blues,0.647,0.876
8165,52MCmoSCtPRbVN5Njdo6G5,Run Rudolph Run,blues,0.647,0.876
8166,7m4luTtlene5vS6xUvWxRt,Run Rudolph Run,blues,0.647,0.876
8168,3RXAcz7Sa6JDZSxcH1EEQ6,Run Rudolph Run,blues,0.647,0.876
8172,7pO8TsOPKQCgFVLYkoCiFV,Run Rudolph Run,blues,0.647,0.876
8173,0GZpxW1UfoInSJug7m34fR,Run Rudolph Run,blues,0.647,0.876
8174,0qyMrxLDdU14grB9GWndxa,Run Rudolph Run,blues,0.647,0.876
8175,0UfbRLBZi1780FbGc07dg0,Run Rudolph Run,blues,0.647,0.876
8176,1Jc5rVFIyOy6XO3ATB1Lh9,Run Rudolph Run,blues,0.647,0.876


In [29]:
tracks_unique = tracks.drop_duplicates(subset="key", keep="first").reset_index(drop=True)
print(len(tracks_unique), "unique tracks ready for joining")


77414 unique tracks ready for joining


In [30]:
pairs = pairs.copy()
pairs["key"] = pairs["artistname"].map(normalise) + "|" + pairs["trackname"].map(normalise)

matched_keys = set(tracks_unique["key"])
pairs["matched"] = pairs["key"].isin(matched_keys)

print("unique songs:", len(pairs))
print("matched:", pairs["matched"].sum())
print("match rate:", round(100 * pairs["matched"].mean(), 2), "%")

unique songs: 2823960
matched: 57117
match rate: 2.02 %


In [31]:
plays = plays.merge(
    pairs[["artistname", "trackname", "key", "matched"]],
    on=["artistname", "trackname"],
    how="left"
)

print("total interactions:", len(plays))
print("interactions kept:", int(plays["matched"].sum()))
print("coverage:", round(100 * plays["matched"].mean(), 2), "%")

total interactions: 12901979
interactions kept: 1531635
coverage: 11.87 %


In [32]:
data = plays[plays["matched"]].copy()

print("interactions:", len(data))
print("users:", data["user_id"].nunique())
print("songs:", data["key"].nunique())
print("avg interactions per user:", round(len(data) / data["user_id"].nunique(), 1))

interactions: 1531635
users: 14857
songs: 21219
avg interactions per user: 103.1


In [33]:
data = plays[plays["matched"]].copy()

print("interactions:", len(data))
print("users:", data["user_id"].nunique())
print("songs:", data["key"].nunique())
print("avg interactions per user:", round(len(data) / data["user_id"].nunique(), 1))


interactions: 1531635
users: 14857
songs: 21219
avg interactions per user: 103.1


In [ ]:
plays = plays.merge(
    pairs[["artistname", "trackname", "key", "matched"]],
    on=["artistname", "trackname"],
    how="left"
)

print("total interactions:", len(plays))
print("interactions kept:", int(plays["matched"].sum()))
print("coverage:", round(100 * plays["matched"].mean(), 2), "%")

total interactions: 12901979
interactions kept: 1531635
coverage: 11.87 %


In [36]:
data.to_pickle("../data/interactions_clean.pkl")
print("saved")

saved


In [2]:
import pandas as pd
data = pd.read_pickle("../data/interactions_clean.pkl")
print(len(data))

1531635


In [5]:
print("rows:", len(data))
print("unique user-song pairs:", len(data[["user_id", "key"]].drop_duplicates()))

rows: 1531635
unique user-song pairs: 1209024


In [3]:
interactions = (
    data.groupby(["user_id", "key"])
        .size()
        .reset_index(name="strength")
)

print(interactions.shape)
print(interactions["strength"].describe())
interactions.head()

(1209024, 3)
count    1.209024e+06
mean     1.266836e+00
std      7.176235e-01
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      6.800000e+01
Name: strength, dtype: float64


,user_id,key,strength
0,00055176fea33f6e027cd3302289378b,all time low|dear maria count me in,1
1,00055176fea33f6e027cd3302289378b,all time low|merry christmas kiss my ass,1
2,00055176fea33f6e027cd3302289378b,all time low|therapy,1
3,00055176fea33f6e027cd3302289378b,becky g|shower,1
4,00055176fea33f6e027cd3302289378b,blink 182|after midnight,1


In [4]:
from scipy.sparse import csr_matrix

users = interactions["user_id"].unique()
songs = interactions["key"].unique()

user_to_row = {u: i for i, u in enumerate(users)}
song_to_col = {s: i for i, s in enumerate(songs)}

rows = interactions["user_id"].map(user_to_row)
cols = interactions["key"].map(song_to_col)

matrix = csr_matrix(
    (interactions["strength"], (rows, cols)),
    shape=(len(users), len(songs))
)

print("shape:", matrix.shape)
print("filled cells:", matrix.nnz)
print("density:", round(100 * matrix.nnz / (matrix.shape[0] * matrix.shape[1]), 3), "%")

shape: (14857, 21219)
filled cells: 1209024
density: 0.384 %


In [5]:
per_user = np.diff(matrix.indptr)

print("min songs per user:", per_user.min())
print("median:", int(np.median(per_user)))
print("max:", per_user.max())
print()
print("users with fewer than 5 songs:", (per_user < 5).sum())
print("users with fewer than 10 songs:", (per_user < 10).sum())

min songs per user: 1
median: 45
max: 2083

users with fewer than 5 songs: 1626
users with fewer than 10 songs: 2805


In [6]:
K_HOLDOUT = 5
MIN_FOR_EVAL = 10
rng = np.random.default_rng(42)

eval_users = np.where(per_user >= MIN_FOR_EVAL)[0]

mask = np.ones(matrix.nnz, dtype=bool)
test = {}

for u in eval_users:
    start, end = matrix.indptr[u], matrix.indptr[u + 1]
    positions = rng.choice(np.arange(start, end), size=K_HOLDOUT, replace=False)
    test[u] = matrix.indices[positions]
    mask[positions] = False

rows_all = np.repeat(np.arange(matrix.shape[0]), per_user)
train = csr_matrix(
    (matrix.data[mask], (rows_all[mask], matrix.indices[mask])),
    shape=matrix.shape
)

print("evaluated users:", len(eval_users))
print("train cells:", train.nnz)
print("held out:", matrix.nnz - train.nnz)

evaluated users: 12052
train cells: 1148764
held out: 60260


In [7]:
def precision_at_k(recommended, relevant, k):
    hits = len(set(recommended[:k]) & set(relevant))
    return hits / k

def recall_at_k(recommended, relevant, k):
    hits = len(set(recommended[:k]) & set(relevant))
    return hits / len(relevant)

def ndcg_at_k(recommended, relevant, k):
    rel = set(relevant)
    dcg = sum(1 / np.log2(i + 2) for i, item in enumerate(recommended[:k]) if item in rel)
    ideal = sum(1 / np.log2(i + 2) for i in range(min(len(rel), k)))
    return dcg / ideal if ideal > 0 else 0.0

In [8]:
item_popularity = np.asarray(train.sum(axis=0)).ravel()
popular_ranking = np.argsort(-item_popularity)

def recommend_popular(user_row, k):
    seen = set(train.indices[train.indptr[user_row]:train.indptr[user_row + 1]])
    out = [i for i in popular_ranking if i not in seen]
    return out[:k]

def recommend_random(user_row, k):
    seen = set(train.indices[train.indptr[user_row]:train.indptr[user_row + 1]])
    choices = rng.choice(train.shape[1], size=k + len(seen) + 50, replace=False)
    out = [i for i in choices if i not in seen]
    return out[:k]

In [9]:
def evaluate(recommend_fn, k=10, sample=2000):
    chosen = rng.choice(eval_users, size=min(sample, len(eval_users)), replace=False)
    p, r, n = [], [], []
    for u in chosen:
        recs = recommend_fn(u, k)
        truth = test[u]
        p.append(precision_at_k(recs, truth, k))
        r.append(recall_at_k(recs, truth, k))
        n.append(ndcg_at_k(recs, truth, k))
    return {
        "precision@10": round(np.mean(p), 4),
        "recall@10": round(np.mean(r), 4),
        "ndcg@10": round(np.mean(n), 4),
    }

print("popular:", evaluate(recommend_popular))
print("random: ", evaluate(recommend_random))

popular: {'precision@10': np.float64(0.0104), 'recall@10': np.float64(0.0208), 'ndcg@10': np.float64(0.0188)}
random:  {'precision@10': np.float64(0.0002), 'recall@10': np.float64(0.0004), 'ndcg@10': np.float64(0.0004)}
